# CHARTEVENTS - Silver Layer

This notebook cleans and validates the MIMIC-IV `chartevents` Bronze table before storing it in the Silver layer.

`CHARTEVENTS` contains observations recorded during ICU stays, including measurements, assessments and other bedside charting. Each event can be linked to `D_ITEMS` through `itemid` to understand what was recorded, and where available to `CAREGIVER` through `caregiver_id`.

The Silver layer focuses on missing values, duplicate records, key relationships and basic text consistency while preserving the different numeric and text event values.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

chartevents = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.chartevents_raw"
)

print(f"Rows: {chartevents.count():,}")
print(f"Columns: {len(chartevents.columns)}")

## Data Quality Check

The main event fields are checked for missing values before deciding which Silver transformations are needed.

In [0]:
# Check missing values in relevant fields
check_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "caregiver_id",
    "charttime",
    "storetime",
    "itemid",
    "value",
    "valuenum",
    "valueuom",
    "warning"
]

total = chartevents.count()

for column in check_columns:
    nulls = chartevents.filter(F.col(column).isNull()).count()
    pct = (nulls / total) * 100

    if nulls > 0:
        print(f"{column}: {nulls:,} nulls ({pct:.2f}%)")

### Missing Value Findings

- `caregiver_id` is missing for a small portion of chart events, so not every event can be linked back to a caregiver.
- `valuenum` and `valueuom` are missing frequently because many charted observations are text or categorical rather than numeric measurements.
- `storetime` and `warning` are only missing for a very small number of records.
- These values should be preserved as nulls rather than filled without a clear clinical reason.

In [0]:
# Check duplicates and lookup relationships
duplicate_rows = (
    chartevents
    .groupBy(
        "subject_id",
        "hadm_id",
        "stay_id",
        "caregiver_id",
        "charttime",
        "itemid",
        "value"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

d_items = spark.table("workspace.silver.d_items")
caregiver = spark.table("workspace.silver.caregiver")

unmatched_items = (
    chartevents
    .select("itemid")
    .distinct()
    .join(d_items.select("itemid"), on="itemid", how="left_anti")
    .count()
)

unmatched_caregivers = (
    chartevents
    .filter(F.col("caregiver_id").isNotNull())
    .select("caregiver_id")
    .distinct()
    .join(caregiver.select("caregiver_id"), on="caregiver_id", how="left_anti")
    .count()
)

print(f"Potential duplicate event groups: {duplicate_rows:,}")
print(f"Unmatched itemid values: {unmatched_items:,}")
print(f"Unmatched caregiver_id values: {unmatched_caregivers:,}")

### Data Quality Findings

- No duplicate event groups were found.
- All `itemid` values successfully match `D_ITEMS`.
- All populated `caregiver_id` values successfully match `CAREGIVER`.
- The main missing values appear to be expected for different types of charted observations rather than broken relationships.

Based on this, the Silver cleaning can stay fairly light and preserve the original event structure.

In [0]:
# Clean text fields
chartevents_silver = (
    chartevents
    .withColumn(
        "value",
        F.when(F.trim(F.col("value")) == "", None)
         .otherwise(F.trim(F.col("value")))
    )
    .withColumn(
        "valueuom",
        F.when(F.trim(F.col("valueuom")) == "", None)
         .otherwise(F.trim(F.col("valueuom")))
    )
)

print(f"CHARTEVENTS Silver: {chartevents_silver.count():,} rows")
display(chartevents_silver.limit(10))

### Silver Transformation Check

`CHARTEVENTS` is preserving the different ways ICU observations are recorded, including text values, numeric values and measurement units.

The cleaned output looks consistent, with optional numeric and unit fields remaining null where they do not apply.

In [0]:
# Save Silver table
(
    chartevents_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.chartevents")
)

bronze_count = spark.table("workspace.bronze.chartevents_raw").count()
silver_count = spark.table("workspace.silver.chartevents").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")

### Silver Result

`CHARTEVENTS` was successfully written to the Silver layer with all Bronze records preserved.

The table now has light text standardisation, validated lookup relationships and no duplicate event groups, while keeping meaningful null values unchanged.